In [ ]:
# week 3  .

In [1]:
import os
import json
import csv
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialize Clients (OpenRouter + Ollama)
load_dotenv(override=True)

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

# Ollama local model
ollama_client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

OLLAMA_MODEL = "llama3.2"

# OpenRouter model
if openrouter_api_key:
    openrouter_client = OpenAI(
        api_key=openrouter_api_key,
        base_url="https://openrouter.ai/api/v1"
    )
    OPENROUTER_MODEL = "openai/gpt-4o-mini"
else:
    openrouter_client = None
    OPENROUTER_MODEL = None

print("Ollama model:", OLLAMA_MODEL)
print("OpenRouter:", "ready" if openrouter_api_key else "no key")

Ollama model: llama3.2
OpenRouter: no key


In [3]:
# Define Schemas
SCHEMAS = {
    "Coding Questions": {
        "description": "Programming questions with explanation.",
        "fields": ["question", "language", "difficulty", "explanation"],
        "hint": "difficulty is beginner, intermediate, or advanced"
    },

    "Python Code Explanation": {
        "description": "Python code snippets with explanation",
        "fields": ["code_snippet", "concept", "explanation"],
        "hint": "concept could be generator, loop, list comprehension"
    },

    "DevOps Troubleshooting": {
        "description": "DevOps troubleshooting scenarios",
        "fields": ["problem", "technology", "solution"],
        "hint": "technology examples: Kubernetes, Docker, Git"
    }
}

In [4]:
# Schema Helper Function
def get_schema_description(schema_choice, custom_text=""):

    if schema_choice == "Custom" and custom_text.strip():
        return custom_text

    s = SCHEMAS.get(schema_choice)

    fields = ", ".join(s["fields"])

    return f"""
Output JSON objects with keys:
{fields}

{s["hint"]}
"""

In [5]:
# Prompt Builder
def build_prompts(schema_description, num_rows):

    system_prompt = """
You are an AI that generates synthetic datasets for machine learning.

Rules:
- Output ONLY valid JSON
- Return a JSON array
- Each item must follow the schema exactly
- Vary wording and style across rows
"""

    user_prompt = f"""
Generate {num_rows} records.

Schema:
{schema_description}

Return only a JSON array.
"""

    return system_prompt, user_prompt

In [6]:
# Dataset Generation Function
def generate_dataset(schema_choice, custom_schema, num_rows, client, model):

    schema_description = get_schema_description(schema_choice, custom_schema)

    system_prompt, user_prompt = build_prompts(schema_description, num_rows)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_prompt}
        ]
    )

    content = response.choices[0].message.content.strip()

    try:
        data = json.loads(content)
        return data, None
    except Exception as e:
        return [], str(e)

In [7]:
# Model Selector
MODEL_OPTIONS = [
    "Ollama (llama3.2)",
    "OpenRouter (gpt-4o-mini)"
]

def get_client_and_model(choice):

    if choice == "OpenRouter (gpt-4o-mini)" and openrouter_client:
        return openrouter_client, OPENROUTER_MODEL

    return ollama_client, OLLAMA_MODEL


In [8]:
# Gradio Generator Function
def run_generate(schema_choice, custom_schema, num_rows, model_choice):

    client, model = get_client_and_model(model_choice)

    data, err = generate_dataset(
        schema_choice,
        custom_schema,
        num_rows,
        client,
        model
    )

    if err:
        return [], err, None

    if not data:
        return [], "No data generated", None

    keys = list(data[0].keys())

    preview = [[row.get(k,"") for k in keys] for row in data]

    fd, path = tempfile.mkstemp(suffix=".csv")

    with os.fdopen(fd, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(data)

    return preview, "", path

In [ ]:
# Gradio UI
SCHEMA_OPTIONS = list(SCHEMAS.keys()) + ["Custom"]

with gr.Blocks(title="AI Coding Dataset Generator") as demo:

    gr.Markdown("# Week 3 – Synthetic Dataset Generator")

    schema_dd = gr.Dropdown(
        SCHEMA_OPTIONS,
        value="Coding Questions",
        label="Dataset Schema"
    )

    custom_schema = gr.Textbox(
        label="Custom schema",
        placeholder="question, answer, difficulty"
    )

    model_dd = gr.Dropdown(
        MODEL_OPTIONS,
        value=MODEL_OPTIONS[0],
        label="Model"
    )

    num_rows = gr.Slider(
        5,
        50,
        value=10,
        step=1,
        label="Number of rows"
    )

    generate_btn = gr.Button("Generate Dataset")

    preview = gr.Dataframe(label="Preview")

    status = gr.Textbox(label="Status")

    download = gr.File(label="Download CSV")

    generate_btn.click(
        run_generate,
        inputs=[schema_dd, custom_schema, num_rows, model_dd],
        outputs=[preview, status, download]
    )

demo.launch()